# **Chapter 12: Networked programs**

## **12.1 Hypertext Transfer Protocol - HTTP**

The network protocol that powers the web is actually quite simple and there is built-in support in Python called socket which makes it very easy to make network connections and retrieve data over those sockets in a Python program.

A socket is much like a file, except that a single socket provides a two-way connection between two programs. You can both read from and write to the same socket. If you write something to a socket, it is sent to the application at the other end of the socket. If you read from the socket, you are given the data which the other
application has sent.

A protocol is a set of precise rules that determine who is to go first, what they are
to do, and then what the responses are to that message, and who sends next, and
so on.

## **12.2 The world's simplest web browser**

First the program makes a connection to port 80 on the server www.pr4e.com.
Since our program is playing the role of the “web browser”, the HTTP protocol
says we must send the GET command followed by a blank line. \r\n signifies
an EOL (end of line), so \r\n\r\n signifies nothing between two EOL sequences.
That is the equivalent of a blank line.

Once we send that blank line, we write a loop that receives data in 512-character
chunks from the socket and prints the data out until there is no more data to read
(i.e., the recv() returns an empty string).

In [7]:
import socket

# 1. Crear el Socket, es como un "enchufe" que conecta el programa con otra red.
mysock = socket.socket(socket.AF_INET,socket.SOCK_STREAM)

# 2. Conectarse al servidor 'data.pr4e.org' en el puerto 80, HTTP
mysock.connect(('data.pr4e.org', 80))

# 3. Enviar la petición HTTP:
#    GET -> método para pedir un recurso.
#    http://data.pr4e.org/romeo.txt -> recurso solicitado.
#    HTTP/1.0 -> versión del protocolo
cmd = 'GET http://data.pr4e.org/romeo.txt HTTP/1.0\r\n\r\n'.encode()
mysock.send(cmd)

# 4. Recibir la respuesta
while True:
    data = mysock.recv(512)
    if len(data) < 1:
        break
    print(data.decode(), end = '')

# 5. Cerrar la conexión
mysock.close()

HTTP/1.1 200 OK
Date: Tue, 14 Apr 2026 15:17:18 GMT
Server: Apache/2.4.52 (Ubuntu)
Last-Modified: Sat, 13 May 2017 11:22:22 GMT
ETag: "a7-54f6609245537"
Accept-Ranges: bytes
Content-Length: 167
Cache-Control: max-age=0, no-cache, no-store, must-revalidate
Pragma: no-cache
Expires: Wed, 11 Jan 1984 05:00:00 GMT
Connection: close
Content-Type: text/plain

But soft what light through yonder window breaks
It is the east and Juliet is the sun
Arise fair sun and kill the envious moon
Who is already sick and pale with grief


After the server sends us the headers, it adds a blank line to indicate the end of
the headers, and then sends the actual data of the file *romeo.txt.*

This example shows how to make a low-level network connection with sockets.
Sockets can be used to communicate with a web server or with a mail server or
many other kinds of servers. All that is needed is to find the document which
describes the protocol and write the code to send and receive the data according
to the protocol.

One of the requirements for using the HTTP protocol is the need to send and
receive data as bytes objects, instead of strings. In the preceding example, the
`encode()` and `decode()` methods convert strings into bytes objects and back again.

## **12.3 Retrieving an image over HTTP**

In the above example, we retrieved a plain text file which had newlines in the file and we simply copied the data to the screen as the program ran. We can use a
similar program to retrieve an image using HTTP.

In [11]:
import socket
import time

HOST = 'data.pr4e.org'
PORT = 80

mysock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
mysock.connect((HOST, PORT))

cmd = b'GET http://data.pr4e.org/cover3.jpg HTTP/1.0\r\n\r\n'
mysock.sendall(cmd)

count = 0
picture = b''

while True:
    data = mysock.recv(5120)
    if len(data) < 1: break
    # time.sleep(0.25)
    count = count + len(data)
    print(len(data), count)
    picture = picture + data

mysock.close()

pos = picture.find(b'\r\n\r\n')
print('Header length', pos)
print(picture[:pos].decode())

picture = picture[pos + 4:]
fhand = open('stuff.jpg', 'wb')
fhand.write(picture)
fhand.close()

5120 5120
320 5440
4080 9520
4080 13600
5120 18720
5120 23840
640 24480
2720 27200
1360 28560
1360 29920
5120 35040
1680 36720
5120 41840
3040 44880
2720 47600
4080 51680
5120 56800
5120 61920
640 62560
1360 63920
2720 66640
5120 71760
5120 76880
3360 80240
5120 85360
3040 88400
2720 91120
5120 96240
5120 101360
3360 104720
5120 109840
320 110160
2720 112880
2720 115600
2720 118320
5120 123440
320 123760
5120 128880
5120 134000
2000 136000
1360 137360
2720 140080
2720 142800
2720 145520
2720 148240
2720 150960
5120 156080
5120 161200
640 161840
2720 164560
2720 167280
2720 170000
2720 172720
4080 176800
2720 179520
5120 184640
1680 186320
4080 190400
2720 193120
5120 198240
5120 203360
2000 205360
5120 210480
320 210800
2720 213520
2720 216240
2720 218960
2720 221680
5120 226800
3808 230608
Header length 394
HTTP/1.1 200 OK
Date: Tue, 14 Apr 2026 15:26:46 GMT
Server: Apache/2.4.52 (Ubuntu)
Last-Modified: Mon, 15 May 2017 12:27:40 GMT
ETag: "38342-54f8f2e5b6277"
Accept-Ranges: bytes
Con

Your results may be different depending on your network speed.
Also note that on the last call to `recv()` we get 3167 bytes, which is the end of the stream, and in the next call to `recv()` we get a zero‑length string that tells us that the server has called `close()` on its end of the socket and there is no more data forthcoming.

We can slow down our successive `recv()` calls by uncommenting the call to `time.sleep()`.
This way, we wait a quarter of a second after each call so that the server can “get ahead” of us and send more data to us before we call `recv()` again.

With the delay in place the program executes as follows...

In [14]:
import socket
import time

HOST = 'data.pr4e.org'
PORT = 80

mysock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
mysock.connect((HOST, PORT))

cmd = b'GET http://data.pr4e.org/cover3.jpg HTTP/1.0\r\n\r\n'
mysock.sendall(cmd)

count = 0
picture = b''

while True:
    data = mysock.recv(5120)
    if len(data) < 1: break
    time.sleep(0.25)
    count = count + len(data)
    print(len(data), count)
    picture = picture + data

mysock.close()

pos = picture.find(b'\r\n\r\n')
print('Header length', pos)
print(picture[:pos].decode())

picture = picture[pos + 4:]
fhand = open('stuff.jpg', 'wb')
fhand.write(picture)
fhand.close()

4080 4080
5120 9200
5120 14320
5120 19440
5120 24560
5120 29680
5120 34800
5120 39920
5120 45040
5120 50160
5120 55280
5120 60400
5120 65520
5120 70640
5120 75760
5120 80880
5120 86000
5120 91120
5120 96240
5120 101360
5120 106480
5120 111600
5120 116720
5120 121840
5120 126960
5120 132080
5120 137200
5120 142320
5120 147440
5120 152560
5120 157680
5120 162800
5120 167920
5120 173040
5120 178160
5120 183280
5120 188400
5120 193520
5120 198640
5120 203760
5120 208880
5120 214000
5120 219120
5120 224240
5120 229360
1248 230608
Header length 394
HTTP/1.1 200 OK
Date: Tue, 14 Apr 2026 15:55:03 GMT
Server: Apache/2.4.52 (Ubuntu)
Last-Modified: Mon, 15 May 2017 12:27:40 GMT
ETag: "38342-54f8f2e5b6277"
Accept-Ranges: bytes
Content-Length: 230210
Vary: Accept-Encoding
Cache-Control: max-age=0, no-cache, no-store, must-revalidate
Pragma: no-cache
Expires: Wed, 11 Jan 1984 05:00:00 GMT
Connection: close
Content-Type: image/jpeg


## **12.4 Retrieving web pages with `urllib`**

While we can manually send and receive data over HTTP using the socket library,
there is a much simpler way to perform this common task in Python by using the urllib library.

Using urllib, you can treat a web page much like a file.
You simply indicate which web page you would like to retrieve and urllib handles all of the HTTP protocol and header details.

In [15]:
import urllib.request

fhand = urllib.request.urlopen('http://data.pr4e.org/romeo.txt')
for line in fhand:
    print(line.decode().strip())

But soft what light through yonder window breaks
It is the east and Juliet is the sun
Arise fair sun and kill the envious moon
Who is already sick and pale with grief


Once the web page has been opened with `urllib.request.urlopen`, we can treat
it like a file and read through it using a for loop.
    
When the program runs,we only see the output of the contents of the file. The
headers are still sent,but the `urllib` code consumes the headers and only returns
the data to us.

As an example, we can write a program to retrieve the data for `romeo.txt` and
compute the frequency of each word in the file as follows:

In [16]:
import urllib.request

fhand = urllib.request.urlopen('http://data.pr4e.org/romeo.txt')
counts = dict()

for line in fhand:
    words = line.decode().strip().split()
    for word in words:
        counts[word] = counts.get(word, 0) + 1

print(counts)

{'But': 1, 'soft': 1, 'what': 1, 'light': 1, 'through': 1, 'yonder': 1, 'window': 1, 'breaks': 1, 'It': 1, 'is': 3, 'the': 3, 'east': 1, 'and': 3, 'Juliet': 1, 'sun': 2, 'Arise': 1, 'fair': 1, 'kill': 1, 'envious': 1, 'moon': 1, 'Who': 1, 'already': 1, 'sick': 1, 'pale': 1, 'with': 1, 'grief': 1}


## **12.5 Reading binary files using `urllib`**

Sometimes you want to retrieve a non-text (or binary) file such as an image or
video file. The data in these files is generally not useful to print out, but you can
easily make a copy of a URL to a local file on your hard disk using `urllib`.

The pattern is to open the `URL` and use `read` to download the entire contents of
the document into a string variable (`img`) then write that information to a local
file as follows:

In [17]:
import urllib.request, urllib.parse, urllib.error

img = urllib.request.urlopen('http://data.pr4e.org/cover3.jpg').read()
fhand = open('cover3.jpg', 'wb')
fhand.write(img)
fhand.close()

This program reads all of the data in at once across the network and stores it in the
variable img in the main memory of your computer, then opens the file `cover.jpg` and writes the data out to your disk. The `wb` argument for `open()` opens a binary file for writing only. This program will work if the size of the file is less than the size of the memory of your computer.

## **12.6 Parsing HTML and scraping the web**

One of the common uses of the urllib capability in Python is to scrape the web.
Web scraping is when we write a program that pretends to be a web browser and
retrieves pages, then examines the data in those pages looking for patterns.
    
As an example, a search engine such as Google will look at the source of one web
page and extract the links to other pages and retrieve those pages, extracting links,
and so on. Using this technique, Google spiders its way through nearly all of the
pages on the web.

Google also uses the frequency of links from pages it finds to a particular page as
one measure of how “important” a page is and how high the page should appear
in its search results.

## **12.7 Parsing HTML using regular expressions**

One simple way to parse HTML is to use regular expressions to repeatedly search
for and extract substrings that match a particular pattern.


We can construct a well-formed regular expression to match and extract the link
values from the above text as follows

Our regular expression looks for strings that start with “href="http://” or
“href="https://”, followed by one or more characters (.+?), followed by another
double quote. The question mark behind the [s]? indicates to search for the
string “http” followed by zero or one “s”.

In [20]:
# Search for link values within URL input
import urllib.request, urllib.parse, urllib.error
import re
import ssl

# Ignore SSL certificate errors
ctx = ssl.create_default_context() # Crea un contexto SSL/TLS, que define cómo se manejara la conexión
ctx.check_hostname = False # Se desactivan esas verificaciones para ignorar errores de certificados
ctx.verify_mode = ssl.CERT_NONE

url = input('Enter - ')
html = urllib.request.urlopen(url, context = ctx).read()
links = re.findall(b'href="(http[s]?://.*?)"', html)
for link in links:
    print(link.decode())

Enter -  https://docs.python.org


https://docs.python.org/3/index.html
https://www.python.org/
https://docs.python.org/3.15/
https://docs.python.org/3.14/
https://docs.python.org/3.13/
https://docs.python.org/3.12/
https://docs.python.org/3.11/
https://docs.python.org/3.10/
https://docs.python.org/3.9/
https://docs.python.org/3.8/
https://docs.python.org/3.7/
https://docs.python.org/3.6/
https://docs.python.org/3.5/
https://docs.python.org/3.4/
https://docs.python.org/3.3/
https://docs.python.org/3.2/
https://docs.python.org/3.1/
https://docs.python.org/3.0/
https://docs.python.org/2.7/
https://docs.python.org/2.6/
https://www.python.org/doc/versions/
https://peps.python.org/
https://wiki.python.org/moin/BeginnersGuide
https://wiki.python.org/moin/PythonBooks
https://www.python.org/doc/av/
https://devguide.python.org/
https://www.python.org/
https://devguide.python.org/documentation/help-documenting/
https://docs.python.org/3.15/
https://docs.python.org/3.14/
https://docs.python.org/3.13/
https://docs.python.org/3.12/


## **12.8 Parsing HTML using BeautifulSoup**

There are a number of Python libraries which can help you parse HTML and
extract data from the pages. Each of the libraries has its strengths and weaknesses
and you can pick one based on your needs.
    
As an example, we will simply parse some HTML input and extract links using
the BeautifulSoup library. BeautifulSoup tolerates highly flawed HTML and still
lets you easily extract the data you need. You can download and install the
BeautifulSoup code from:

In [21]:
import urllib.request, urllib.parse, urllib.error
from bs4 import BeautifulSoup
import ssl

ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

url = input('Enter - ')
html = urllib.request.urlopen(url, context = ctx).read()
soup = BeautifulSoup(html, 'html.parser')

tags = soup('a')
for tag in tags:
    print(tag.get('href', None))

Enter -  https://docs.python.org


https://www.python.org/
download.html
https://docs.python.org/3.15/
https://docs.python.org/3.14/
https://docs.python.org/3.13/
https://docs.python.org/3.12/
https://docs.python.org/3.11/
https://docs.python.org/3.10/
https://docs.python.org/3.9/
https://docs.python.org/3.8/
https://docs.python.org/3.7/
https://docs.python.org/3.6/
https://docs.python.org/3.5/
https://docs.python.org/3.4/
https://docs.python.org/3.3/
https://docs.python.org/3.2/
https://docs.python.org/3.1/
https://docs.python.org/3.0/
https://docs.python.org/2.7/
https://docs.python.org/2.6/
https://www.python.org/doc/versions/
https://peps.python.org/
https://wiki.python.org/moin/BeginnersGuide
https://wiki.python.org/moin/PythonBooks
https://www.python.org/doc/av/
https://devguide.python.org/
genindex.html
py-modindex.html
https://www.python.org/
#

whatsnew/3.14.html
whatsnew/index.html
tutorial/index.html
library/index.html
reference/index.html
using/index.html
howto/index.html
installing/index.html
distributing/i

## **12.11 Exercises**

### **Exercise 1:**

Change the socket program `socket1.py` to prompt the user for the URL so it can read any web page.

You can use `split('/')` to break the URL into its component parts so you can extract the host name for the socket `connect` call. Add error checking using `try` and `except` to handle the condition where the user enters an improperly formatted or non-existent URL.

In [35]:
import socket
import re

try:
    url = input('Enter - ')
    
    url = url.strip()
    server = re.findall('http[s]?://([^/]*)/?.*', url)[0].strip()
    print('URL: ', url)
    print('SERVER: ', server)
    
    mysock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    mysock.connect((server, 80))
    
    cmd = 'GET ' + url + ' HTTP/1.0\r\n\r\n'
    print(cmd)
    cmd = cmd.encode()
    mysock.send(cmd)
except:
    print('Improper formatted or non-existent URL')

while True:
    data = mysock.recv(512)
    if len(data) < 1: break
    print(data.decode(), end = "")

mysock.close()

Enter -   http://data.pr4e.org/romeo.txt 


URL:  http://data.pr4e.org/romeo.txt
SERVER:  data.pr4e.org
GET http://data.pr4e.org/romeo.txt HTTP/1.0


HTTP/1.1 200 OK
Date: Tue, 14 Apr 2026 17:34:56 GMT
Server: Apache/2.4.52 (Ubuntu)
Last-Modified: Sat, 13 May 2017 11:22:22 GMT
ETag: "a7-54f6609245537"
Accept-Ranges: bytes
Content-Length: 167
Cache-Control: max-age=0, no-cache, no-store, must-revalidate
Pragma: no-cache
Expires: Wed, 11 Jan 1984 05:00:00 GMT
Connection: close
Content-Type: text/plain

But soft what light through yonder window breaks
It is the east and Juliet is the sun
Arise fair sun and kill the envious moon
Who is already sick and pale with grief


### **Exercise 2:**

 Change your socket program so that it counts the number of charac
ters it has received and stops displaying any text after it has shown 3000 characters.
The program should retrieve the entire document and count the total number of
characters and display the count of the number of characters at the end of the
document.

In [50]:
import socket
import re

try:
    url = input('Enter - ')
    url = url.strip()

    host = re.findall('http[s]?://([^/]*)/?.*', url)[0].strip()
    mysock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    mysock.connect((host, 80))

    cmd = 'GET ' + url + ' HTTP/1.0\r\n\r\n'
    cmd = cmd.encode()
    mysock.send(cmd)
except:
    print('Improper formatted or non-existent URL')

count = 0
shown = 0
limit = 50
while True:
    data = mysock.recv(512)
    if len(data) < 1: break
    count += len(data)

    if shown < limit:
        text = data.decode()
        remaining = limit - shown
        print(text[:remaining], end = '')
        shown += len(text[:remaining])

print('\n\nTotal Count: ', count)

mysock.close()

Enter -  http://data.pr4e.org/romeo.txt


HTTP/1.1 200 OK
Date: Tue, 14 Apr 2026 18:08:18 G

Total Count:  536
